# Mouse surfaceome publisher table and installed measurement equivalence


Frozen scope: exact acquired CSPA publisher workbook, existing surface_repertoire loader and installed mouse fields. A temporary source binding preserves the original published filename; no archive rename or runtime modification. Detection-negative means nondetection under this capture protocol, not proven biological absence.

In [1]:
import json,tempfile
from pathlib import Path
from unittest.mock import patch
import numpy as np,pandas as pd
from starplast import host,organisms as O,paths
from starplast.provenance import SourceFile
p=Path("results/host_surfaceome_recovery_2026_10_07/table")
review=json.loads((p/"table_review.json").read_text())
source=SourceFile(**review["file"])
source.verify()
original=Path(source.path)
with tempfile.TemporaryDirectory() as temporary:
    folder=Path(temporary)/"host"/host.CSPA[0]/host.CSPA[1]
    folder.mkdir(parents=True)
    (folder/original.name).symlink_to(original)
    with patch.object(host,"CSPA",(host.CSPA[0],host.CSPA[1],original.name)):
        reproduced=host.surface_repertoire(temporary).set_index("host_id")
installed=pd.read_parquet(paths.cache_file(O.HOST_TABLES[O.MOUSE])).set_index("host_id")
checks={}
for column in ("bmdm_surface_detected","bmdm_surface_intensity"):
    expected=installed[column].dropna()
    missing=sorted(set(expected.index)-set(reproduced.index))
    assert not missing
    observed=reproduced[column].reindex(expected.index)
    if column.endswith("detected"):
        assert observed.notna().all() and observed.eq(expected).all()
        delta=0
    else:
        delta=float((observed-expected).abs().max())
        assert delta==0
    checks[column]={"installed_nonmissing":len(expected),"lost_ids":missing,"max_abs_difference":delta}
record={"source_id":review["source_id"],"file":review["file"],"published_filename":original.name,"legacy_filename":host.CSPA[2],"source_accessions":len(reproduced),"detected_accessions":int(reproduced.bmdm_surface_detected.sum()),"quantified_accessions":int(reproduced.bmdm_surface_intensity.notna().sum()),"installed_reproduction":checks,"negative_semantics":"not_detected_in_this_capture_protocol; not_proven_biological_absence","mapping_scope":"exact_accession_join; historical_accession/reference_mapping_and_unmapped_sources_require_review","runtime_changes":"none"}
(p/"equivalence.json").write_text(json.dumps(record,indent=2)+chr(10))
print(record)

macrophage surfaceome: 150 of 1,296 mouse surface proteins on BMDM (147 with an intensity, 12 where the two sheets disagree)
{'source_id': 'host_macrophage_surfaceome', 'file': {'path': '/media/carruthers/mnt3/claude/toxoplasma_projects/datasets/publisher_review_inputs/host_macrophage_surfaceome/pone.0121314.s002.xlsx', 'sha256': '7502646155bf3f17e8c9b2a1edfbe9f27a169f83534baebe8d4c2e9a7206bf5a', 'bytes': 2721208, 'role': 'processed_input', 'url': 'https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0121314.s002&type=supplementary', 'association': 'exact_registry_resource_URL_and_primary_redirect_filename; legacy_table_transform_unverified'}, 'published_filename': 'pone.0121314.s002.xlsx', 'legacy_filename': 'S1_File.xlsx', 'source_accessions': 1296, 'detected_accessions': 150, 'quantified_accessions': 147, 'installed_reproduction': {'bmdm_surface_detected': {'installed_nonmissing': 1296, 'lost_ids': [], 'max_abs_difference': 0}, 'bmdm_surface_intensity': {'installed